# Source views, likes, and comments

Counts describe the original YouTube video, not the individual clip. Start with archived metadata; optionally retrieve one source.

In [ ]:
from datetime import datetime, timezone
import hashlib
import json
import os
from pathlib import Path
import sqlite3

from dbp_metrics.source_statistics import read_catalog, catalog_record, youtube_record

CATALOG_PATH = Path(os.environ.get("DBP_CATALOG", "/path/to/catalog.sqlite3"))
RUN_LIVE = False


## 1. Read one source

Set the catalog path above. Archived views have an unknown collection date; absent likes and comments stay `None`.

In [ ]:
sources = read_catalog(CATALOG_PATH)
source = sources[0]
with CATALOG_PATH.open("rb") as handle:
    catalog_sha256 = hashlib.file_digest(handle, "sha256").hexdigest()
record = catalog_record(source, catalog_sha256=catalog_sha256)
print(json.dumps(record, indent=2))


## 2. Optionally retrieve current counts

Install the root package's `youtube` extra in your kernel first. Set `RUN_LIVE = True` to make one metadata-only request. No media is downloaded; extractor-reported counts may be rounded.

In [ ]:
if RUN_LIVE:
    from dbp_metrics.cli.youtube_sources import fetch_youtube
    information, version = fetch_youtube(source["source_id"])
    record = youtube_record(source["source_id"], information,
                            observed_at=datetime.now(timezone.utc).isoformat(),
                            extractor_version=version)
    print(json.dumps(record, indent=2))


## 3. See which clips inherit these counts

Every clip below shares this source's counts. Website filters compare these numeric values; histograms count clips, not unique sources.

In [ ]:
database = sqlite3.connect(CATALOG_PATH.resolve().as_uri() + "?mode=ro", uri=True)
try:
    clips = database.execute("SELECT DISTINCT clip_id FROM clips WHERE source_id=? ORDER BY clip_id LIMIT 5",
                             (source["source_id"],)).fetchall()
finally:
    database.close()

for (clip_id,) in clips:
    print(clip_id, record["metrics"])

views = record["metrics"]["source_view_count"]
print("Source views > 1,000:", views is not None and views > 1000)
